# Enarau Objective 1 — Dynamic World Historical Change Detection

Component D (historical LULC change) of the Enarau Conservancy remote-sensing consultancy. Builds Dynamic World V1 (`GOOGLE/DYNAMICWORLD/V1`) probability composites for a buffered union of Enarau, Mbokishi, and
both corridor phases, derives a habitat classification and conversion-pressure scheme, computes
baseline / pre-establishment / current transitions.

Requires a `.env` file in the repo root with `EE_PROJECT=<your-gee-cloud-project-id>` and Earth Engine authenticated on this machine (`earthengine authenticate`).

## 1. Setup

In [ ]:
import ee
import eetools
import geemap

from eetools.compositing import build_composite
from eetools.constants import DW_PROBABILITY_BANDS, DW_SCALE
from eetools.io import export_image_list_to_drive, export_table_to_drive
from eetools.sensors.dynamicworld.preprocessing import get_dynamic_world_collection
from eetools.vectors import get_sites_geometry, vector_files_to_feature_collection
from eetools.visualization.vis_params import SITES_VIS_PARAMS
from eetools.zonal import image_collection_to_region_stats_fc

In [ ]:
try:
    import config
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "config module not found -- run `uv sync` from the repo root to install this "
        "project (including config.py) into the environment."
    ) from exc

In [ ]:
eetools.initialize()

In [ ]:
Map = geemap.Map(height="600px")
Map.add_basemap("SATELLITE")

In [ ]:
# Local aliases for readability -- values are single-sourced in config.py.
CRS = config.PROJECT_CRS
SCALE = DW_SCALE
EXPORT_FOLDER = config.DW_EXPORT_FOLDER
DERIVED_BANDS = config.DW_DERIVED_BANDS
PERIODS = config.DW_PERIODS
SEASONS = config.DW_SEASONS

# Dynamic World's 9th class (snow_and_ice) is excluded per config.DW_EXCLUDED_PROB_BANDS.
FULL_PROB_BANDS = [b for b in DW_PROBABILITY_BANDS if b not in config.DW_EXCLUDED_PROB_BANDS]

## 2. Study-area boundary & project geometry

In [ ]:
site_files = [(site["path"], site["site_id"], site["site_name"]) for site in config.SITES]
sites_fc = vector_files_to_feature_collection(site_files)

# Buffered union of all sites -- one project-wide geometry for every composite/export below,
# not four per-site clips.
project_geom = get_sites_geometry(sites_fc).buffer(config.STUDY_AREA_BUFFER_M, maxError=10)

## 3. Dynamic World collection

In [ ]:
dw_start = ee.Date.fromYMD(config.DW_YEAR_START, 1, 1)
dw_end = ee.Date.fromYMD(config.DW_YEAR_END + 1, 1, 1)

# cover_types=None keeps every probability band plus `label` -- no class filtering, since the
# habitat classification below is built from thresholded probability composites, not `label`.
dw_col = get_dynamic_world_collection(project_geom, dw_start, dw_end)

## 4. Composite, classification, and transition helpers

In [ ]:
def get_date_window(year_start: int, year_end: int, season: str) -> tuple[ee.Date, ee.Date]:
    """Return the (start, end) ee.Date window for a season spanning year_start..year_end.

    A single year is a window with year_start == year_end; a multi-year period (e.g. the
    baseline/pre/current comparisons) uses year_start < year_end.
    """
    if season == "wet":
        return ee.Date.fromYMD(year_start, 3, 1), ee.Date.fromYMD(year_end, 6, 1)
    if season == "dry":
        return ee.Date.fromYMD(year_start, 7, 1), ee.Date.fromYMD(year_end, 11, 1)
    if season == "annual":
        return ee.Date.fromYMD(year_start, 1, 1), ee.Date.fromYMD(year_end + 1, 1, 1)
    raise ValueError("season must be one of: wet, dry, annual")

In [ ]:
def build_habitat_composite(collection: ee.ImageCollection, start: ee.Date, end: ee.Date) -> ee.Image:
    """Median Dynamic World probability composite plus derived habitat proxy bands.

    A median composite (robust to outlier acquisitions) with
    woody/grass/natural/conversion-pressure proxy bands and a per-pixel valid-observation count.
    """
    subset = collection.filterDate(start, end)
    median = build_composite(subset, FULL_PROB_BANDS, composite_stat="median")
    count = subset.select("grass").count().rename("valid_obs_count").toDouble()

    woody = median.select("trees").add(median.select("shrub_and_scrub")).rename("woody_prob")
    grass_prob = median.select("grass").rename("grass_prob")
    natural = woody.add(grass_prob).rename("natural_prob")
    water_wetland = median.select("water").add(median.select("flooded_vegetation")).rename("water_wetland_prob")
    hard_conversion = median.select("crops").add(median.select("built")).rename("hard_conversion_prob")
    conversion_pressure = hard_conversion.add(median.select("bare")).rename("conversion_pressure_prob")
    bare_degradation = median.select("bare").rename("bare_degradation_prob")
    top1 = median.reduce(ee.Reducer.max()).rename("top1_prob")

    derived = ee.Image.cat(
        [natural, woody, grass_prob, conversion_pressure, hard_conversion, bare_degradation, water_wetland, top1, count]
    )
    return median.addBands(derived).clip(project_geom)

In [ ]:
def classify_habitat(composite: ee.Image, min_obs: int) -> ee.Image:
    """Simplified 9-class habitat scheme from thresholded probability bands.

    Thresholds (config.DW_HABITAT_THRESHOLDS) are starting values pending visual calibration
    against high-resolution imagery.

    Each `.where()` call overwrites pixels matching its own condition, so the LAST rule applied
    wins for any pixel matching more than one. Rules are applied lowest-priority-first (natural
    -> grass -> woody -> bare -> water -> built -> crops) so cropland ends up with the highest
    effective precedence, matching the intended reading order (cropland > built > water >
    bare/degraded > woody > grassland > mixed natural > uncertain) -- e.g. a pixel with both
    crops_prob and natural_prob above their thresholds is cropland, not "mixed natural".
    """
    t = config.DW_HABITAT_THRESHOLDS
    valid = composite.select("valid_obs_count").gte(min_obs)
    woody = composite.select("woody_prob")
    grass = composite.select("grass_prob")
    natural = composite.select("natural_prob")
    crops = composite.select("crops")
    built = composite.select("built")
    bare = composite.select("bare")
    water_wetland = composite.select("water_wetland_prob")
    top1 = composite.select("top1_prob")

    habitat_class = ee.Image(8).rename("habitat_class")  # default: uncertain / low confidence
    # Catch-all: any real natural-vegetation signal not claimed by a more specific rule below.
    # No dominance-margin condition here -- that would leave a dead zone for pixels with real
    # but skewed natural_prob that doesn't clear woody_min/grass_min individually.
    habitat_class = habitat_class.where(natural.gte(t["natural_min"]), 3)
    habitat_class = habitat_class.where(
        grass.gte(t["grass_min"]).And(grass.subtract(woody).gte(t["woody_grass_margin"])), 2
    )
    habitat_class = habitat_class.where(
        woody.gte(t["woody_min"]).And(woody.subtract(grass).gte(t["woody_grass_margin"])), 1
    )
    habitat_class = habitat_class.where(bare.gte(t["bare_min"]).And(bare.gt(crops)).And(bare.gt(built)), 6)
    habitat_class = habitat_class.where(water_wetland.gte(t["water_wetland_min"]), 7)
    habitat_class = habitat_class.where(built.gte(t["built_min"]), 5)
    habitat_class = habitat_class.where(crops.gte(t["crops_min"]), 4)

    return (
        habitat_class.updateMask(valid)
        .updateMask(top1.gte(t["top1_min"]))
        .clip(project_geom)
        .rename("habitat_class")
        .toInt()
    )

In [ ]:
def classify_pressure(composite: ee.Image) -> ee.Image:
    """Low/moderate/high conversion-pressure classes from conversion_pressure_prob."""
    t = config.DW_PRESSURE_THRESHOLDS
    pressure = composite.select("conversion_pressure_prob")
    valid = composite.select("valid_obs_count").gt(0)

    pressure_class = (
        ee.Image(0)
        .rename("pressure_class")
        .where(pressure.gte(t["moderate_min"]).And(pressure.lt(t["high_min"])), 1)
        .where(pressure.gte(t["high_min"]), 2)
        .updateMask(valid)
        .clip(project_geom)
        .toInt()
    )
    return pressure_class

In [ ]:
def make_transition(from_class: ee.Image, to_class: ee.Image) -> ee.Image:
    """Encode a categorical transition as from_class * 10 + to_class (plan §8)."""
    return from_class.multiply(10).add(to_class).rename("transition_code").toInt()

## 5. Annual & seasonal composites (2016-2025)

Builds one composite per (year, season) -- wet, dry, and annual -- for 2016-2025, plus the
habitat and conversion-pressure classifications for each, and a `coverage_flag` band (whether
`valid_obs_count` clears the season-appropriate threshold) used by the zonal stats to
surface valid-pixel coverage without a separate client-side pass. These
should be treated as diagnostic rather than headline reporting products.

In [ ]:
records = {}
for year in range(config.DW_YEAR_START, config.DW_YEAR_END + 1):
    for season in SEASONS:
        min_obs = config.DW_MIN_OBS_ANNUAL if season == "annual" else config.DW_MIN_OBS_SEASONAL
        start, end = get_date_window(year, year, season)
        composite = build_habitat_composite(dw_col, start, end)
        composite = composite.addBands(
            composite.select("valid_obs_count").gte(min_obs).rename("coverage_flag")
        ).set({"year": year, "season": season})
        records[(year, season)] = {
            "composite": composite,
            "habitat_class": classify_habitat(composite, min_obs),
            "pressure_class": classify_pressure(composite),
        }

print(f"Built {len(records)} year/season composites.")

## 6. Headline period composites & transitions

The primary reporting products: baseline (2016-2018), pre-establishment (2019-2021),
and current/post-establishment (2022-2025) composites, classifications, and the two headline
transitions (baseline→current, pre-establishment→current).

In [ ]:
period_records = {}
for period_name, (year_start, year_end) in PERIODS.items():
    start, end = get_date_window(year_start, year_end, "annual")
    composite = build_habitat_composite(dw_col, start, end)
    composite = composite.addBands(
        composite.select("valid_obs_count").gte(config.DW_MIN_OBS_ANNUAL).rename("coverage_flag")
    ).set({"period": period_name})
    period_records[period_name] = {
        "composite": composite,
        "habitat_class": classify_habitat(composite, config.DW_MIN_OBS_ANNUAL),
        "pressure_class": classify_pressure(composite),
    }

transitions = {
    "baseline_to_current": make_transition(
        period_records["baseline"]["habitat_class"], period_records["current"]["habitat_class"]
    ),
    "pre_to_current": make_transition(
        period_records["pre"]["habitat_class"], period_records["current"]["habitat_class"]
    ),
}

### Current-period seasonal composites (Objective 4 input gap-fill)

Objective 4's plan document (§3.1 "Inputs from Objective 1") asks for current-period **wet** and
**dry** seasonal probability/categorical/pressure composites specifically
(`DW_prob_wet_2022_2025_project.tif`, `DW_prob_dry_2022_2025_project.tif`, etc.), in addition to
the annual-type current composite `period_records["current"]` already provides -- `period_records`
only ever builds the `"annual"` window type, so this gap wasn't covered before. Built here to
close it; these feed only the raster exports below (§10c), not the zonal-statistics tables --
scope kept to what was asked, not extended into new stats tables.

In [ ]:
current_start_year, current_end_year = PERIODS["current"]
current_seasonal_records = {}
for season in ("wet", "dry"):
    start, end = get_date_window(current_start_year, current_end_year, season)
    composite = build_habitat_composite(dw_col, start, end)
    composite = composite.addBands(
        composite.select("valid_obs_count").gte(config.DW_MIN_OBS_SEASONAL).rename("coverage_flag")
    ).set({"period": "current", "season": season})
    current_seasonal_records[season] = {
        "composite": composite,
        "habitat_class": classify_habitat(composite, config.DW_MIN_OBS_SEASONAL),
        "pressure_class": classify_pressure(composite),
    }

print(f"Built {len(current_seasonal_records)} current-period seasonal composites.")

## 7. Objective 4 connectivity-ready masks (current period)

Evidence layers only -- not final resistance, source-strength, or corridor products. Built from the current/post-establishment composite per the plan's connectivity-input handoff amendment.

In [ ]:
def build_connectivity_masks(current_stack: ee.Image, current_class: ee.Image) -> dict[str, ee.Image]:
    t = config.DW_CONNECTIVITY_THRESHOLDS
    valid = current_stack.select("valid_obs_count").gte(config.DW_MIN_OBS_ANNUAL).And(
        current_stack.select("top1_prob").gte(t["top1_prob_min"])
    )

    # .selfMask() drops the 0 (non-matching) pixels rather than leaving them as explicit zeros --
    # matches connectivity_analysis_mask's own convention below (ee.Image(1) + updateMask, never
    # an explicit 0) and avoids export_image_to_drive's .unmask(-9999) turning a meaningful "not
    # a source" 0 into ambiguous data instead of proper NoData for downstream R/Circuitscape use.
    natural_habitat_mask = (
        current_class.eq(1)
        .Or(current_class.eq(2))
        .Or(current_class.eq(3))
        .selfMask()
        .rename("natural_habitat_mask")
        .updateMask(valid)
        .toInt()
    )

    high_quality_source_mask = (
        current_stack.select("natural_prob")
        .gte(t["natural_prob_min"])
        .And(current_stack.select("conversion_pressure_prob").lte(t["conversion_pressure_max"]))
        .And(current_stack.select("top1_prob").gte(t["top1_prob_min"]))
        .selfMask()
        .rename("high_quality_source_mask")
        .updateMask(valid)
        .toInt()
    )

    connectivity_analysis_mask = (
        ee.Image(1).rename("connectivity_analysis_mask").updateMask(valid).clip(project_geom).toInt()
    )

    return {
        "analysis_mask": connectivity_analysis_mask,
        "natural_habitat_mask": natural_habitat_mask,
        "high_quality_source_mask": high_quality_source_mask,
    }


connectivity_masks = build_connectivity_masks(
    period_records["current"]["composite"], period_records["current"]["habitat_class"]
)

## 8. Zonal statistics -- build & export only

Everything below builds `ee.FeatureCollections` and hands them straight to
`export_table_to_drive` -- no `.getInfo()` or pandas conversion happens in this notebook (keeps
server-side load down given the number of rasters this notebook also generates). Download the
exported CSVs from Drive and do the pandas joins/analysis locally, in a separate step.

Per-class area is computed as separate binary-mask-and-sum bands (`class_area_bands` below) rather
than via a grouped reducer, so every output `ee.FeatureCollection` is already flat/wide and
exports cleanly to CSV -- a grouped reducer's nested `"groups"` property doesn't serialize to flat
CSV columns.

In [ ]:
def class_area_bands(class_image: ee.Image, codes: list[int], prefix: str) -> ee.Image:
    """Per-class area (ha) as separate bands via binary masks (wide format, CSV-friendly)."""
    area_ha = ee.Image.pixelArea().divide(10000)
    return ee.Image.cat(
        [area_ha.updateMask(class_image.eq(code)).rename(f"{prefix}_{code}") for code in codes]
    )

In [ ]:
# -- Annual/seasonal: mean probability (+ coverage) and area-by-class tables --------------------
annual_seasonal_prob_collection = ee.ImageCollection(
    [rec["composite"] for rec in records.values()]
)
annual_seasonal_prob_stats_fc = image_collection_to_region_stats_fc(
    collection=annual_seasonal_prob_collection,
    regions_fc=sites_fc,
    bands=DERIVED_BANDS + ["coverage_flag"],
    scale=SCALE,
    image_properties=["year", "season"],
    tile_scale=4,
)

annual_seasonal_area_collection = ee.ImageCollection(
    [
        class_area_bands(rec["habitat_class"], config.DW_HABITAT_CLASS_CODES, "habitat_area_ha")
        .addBands(class_area_bands(rec["pressure_class"], config.DW_PRESSURE_CLASS_CODES, "pressure_area_ha"))
        .set({"year": year, "season": season})
        for (year, season), rec in records.items()
    ]
)
annual_seasonal_area_bands = [f"habitat_area_ha_{c}" for c in config.DW_HABITAT_CLASS_CODES] + [
    f"pressure_area_ha_{c}" for c in config.DW_PRESSURE_CLASS_CODES
]
annual_seasonal_area_stats_fc = image_collection_to_region_stats_fc(
    collection=annual_seasonal_area_collection,
    regions_fc=sites_fc,
    bands=annual_seasonal_area_bands,
    scale=SCALE,
    reducers=ee.Reducer.sum(),
    image_properties=["year", "season"],
    tile_scale=4,
)

In [ ]:
# -- Period: mean probability (+ coverage) and area-by-class tables -----------------------------
period_prob_collection = ee.ImageCollection([rec["composite"] for rec in period_records.values()])
period_prob_stats_fc = image_collection_to_region_stats_fc(
    collection=period_prob_collection,
    regions_fc=sites_fc,
    bands=DERIVED_BANDS + ["coverage_flag"],
    scale=SCALE,
    image_properties=["period"],
    tile_scale=4,
)

period_area_collection = ee.ImageCollection(
    [
        class_area_bands(rec["habitat_class"], config.DW_HABITAT_CLASS_CODES, "habitat_area_ha")
        .addBands(class_area_bands(rec["pressure_class"], config.DW_PRESSURE_CLASS_CODES, "pressure_area_ha"))
        .set({"period": period_name})
        for period_name, rec in period_records.items()
    ]
)
period_area_stats_fc = image_collection_to_region_stats_fc(
    collection=period_area_collection,
    regions_fc=sites_fc,
    bands=annual_seasonal_area_bands,
    scale=SCALE,
    reducers=ee.Reducer.sum(),
    image_properties=["period"],
    tile_scale=4,
)

In [ ]:
# -- Transitions: area-by-transition-code table ---------------------------------------------
transition_area_collection = ee.ImageCollection(
    [
        class_area_bands(image, config.DW_TRANSITION_CODES, "transition_area_ha").set({"comparison": name})
        for name, image in transitions.items()
    ]
)
transition_area_bands = [f"transition_area_ha_{c}" for c in config.DW_TRANSITION_CODES]
transition_area_stats_fc = image_collection_to_region_stats_fc(
    collection=transition_area_collection,
    regions_fc=sites_fc,
    bands=transition_area_bands,
    scale=SCALE,
    reducers=ee.Reducer.sum(),
    image_properties=["comparison"],
    tile_scale=4,
)

### Export stats tables to Drive

In [ ]:
STATS_TABLES = {
    "dw_mean_probability_by_site_year_season": annual_seasonal_prob_stats_fc,
    "dw_area_by_class_by_site_year_season": annual_seasonal_area_stats_fc,
    "dw_mean_probability_by_site_period": period_prob_stats_fc,
    "dw_area_by_class_by_site_period": period_area_stats_fc,
    "dw_transition_area_by_site": transition_area_stats_fc,
}

for table_name, table_fc in STATS_TABLES.items():
    export_table_to_drive(
        collection=table_fc,
        description=table_name,
        folder=EXPORT_FOLDER,
        fileNamePrefix=f"zonal_statistics/{table_name}",
    )

print(f"Started {len(STATS_TABLES)} table export tasks to Drive folder '{EXPORT_FOLDER}'.")

## 9. Visual QA

Interactive check against the classification and pressure layers before export (plan §13C) --
for a full high-resolution imagery comparison, inspect settlement edges, cultivation expansion,
and woody/grassland boundaries directly against the EE basemap. Use Mbokishi as the
reference-site sanity check (§13D): if it shows conversion trends not visible in imagery, revisit
the thresholds in `config.DW_HABITAT_THRESHOLDS`/`config.DW_PRESSURE_THRESHOLDS` before
interpreting Enarau/corridor results.

In [ ]:
Map = geemap.Map()
Map.centerObject(project_geom, 12)
Map.addLayer(sites_fc.style(**SITES_VIS_PARAMS), {}, "Study sites")
Map.addLayer(period_records["baseline"]["habitat_class"], config.DW_HABITAT_CLASS_VIS, "Baseline habitat class (2016-2018)")
Map.addLayer(period_records["pre"]["habitat_class"], config.DW_HABITAT_CLASS_VIS, "Pre habitat class (2019-2021)")
Map.addLayer(period_records["current"]["habitat_class"], config.DW_HABITAT_CLASS_VIS, "Current habitat class (2022-2025)", False)
Map.addLayer(period_records["current"]["pressure_class"], config.DW_PRESSURE_VIS, "Current conversion pressure (2022-2025)", False)
Map.addLayer(transitions["baseline_to_current"], config.DW_TRANSITION_VIS, "Baseline -> current transition", False)
Map.addLayer(connectivity_masks["analysis_mask"], {"palette": "ffffff"}, "Connectivity analysis mask", False)
Map.addLayer(connectivity_masks["natural_habitat_mask"], {"palette": "00ff00"}, "Natural habitat mask", False)
Map.addLayer(connectivity_masks["high_quality_source_mask"], {"palette": "0000ff"}, "High-quality source habitat mask", False)
Map

## 10. Export rasters to Google Drive

Split into four independently-triggered groups so each batch can be reviewed and checked in the
EE Task Manager before starting the next, rather than firing dozens of export tasks in one burst.
Each group has a "build" cell (safe, just constructs the task list) and a separate "start" cell
(run explicitly). The per-year annual-type group (10b) is optional and deferred by default -- per
Objective 3's own plan document, per-year annual granularity is secondary/QA-level ("trend plots
... interpreted cautiously"), while its wet/dry seasonal contrast (10a) is the primary design
axis -- export 10b only if a later analysis step actually needs individual-year annual maps.

### 10a. Wet & dry seasonal categorical + pressure maps (2016-2025)

Primary design axis per Objective 3's plan (dry season for structural fragmentation, wet season
for grassland recovery) -- always exported.

In [ ]:
EXPORT_TASKS_SEASONAL = []  # (image, file_prefix, scale)

for (year, season), rec in records.items():
    if season == "annual":
        continue  # per-year annual composites are exported separately in 10b (optional/deferred)
    EXPORT_TASKS_SEASONAL.append(
        (rec["habitat_class"], f"seasonal_categorical_DW_class_{season}_{year}_project", SCALE)
    )
    EXPORT_TASKS_SEASONAL.append(
        (rec["pressure_class"], f"conversion_pressure_DW_pressure_{season}_{year}_project", SCALE)
    )

print(f"{len(EXPORT_TASKS_SEASONAL)} wet/dry seasonal export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_SEASONAL, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} wet/dry seasonal export tasks.")

### 10b. Per-year annual categorical + pressure maps (2016-2025) -- optional, deferred

Objective 3's own plan document treats per-year annual granularity as secondary ("useful for
trend plots and QA" but "interpreted cautiously"), not part of its primary wet/dry seasonal
design (10a) or period comparisons (10c). Skipped by default to reduce EECU/server load --
export only if a later analysis step actually needs individual-year annual maps.

In [ ]:
EXPORT_TASKS_ANNUAL_ONLY = []  # (image, file_prefix, scale)

for (year, season), rec in records.items():
    if season != "annual":
        continue
    EXPORT_TASKS_ANNUAL_ONLY.append(
        (rec["habitat_class"], f"annual_categorical/DW_class_{season}_{year}_project", SCALE)
    )
    EXPORT_TASKS_ANNUAL_ONLY.append(
        (rec["pressure_class"], f"conversion_pressure/DW_pressure_{season}_{year}_project", SCALE)
    )

print(f"{len(EXPORT_TASKS_ANNUAL_ONLY)} per-year annual export tasks staged (optional).")

**Manual step, optional** -- only run this if a later analysis step needs individual-year annual maps; review the count above first.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_ANNUAL_ONLY, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} per-year annual export tasks.")

### 10c. Period connectivity-input stacks, categorical, and pressure maps

Includes the annual-type baseline/pre/current period composites, plus the current-period wet/dry
seasonal composites built above (Objective 4 input gap-fill).

In [ ]:
EXPORT_TASKS_PERIODS = []

PERIOD_LABELS = {"baseline": "2016_2018", "pre": "2019_2021", "current": "2022_2025"}
for period_name, rec in period_records.items():
    label = PERIOD_LABELS[period_name]
    EXPORT_TASKS_PERIODS.append(
        (
            rec["composite"].select(DERIVED_BANDS),
            f"connectivity_inputs_DW_connectivity_inputs_{period_name}_{label}_project",
            SCALE,
        )
    )
    
    EXPORT_TASKS_PERIODS.append(
        (rec["habitat_class"], f"connectivity_inputs_DW_class_{period_name}_{label}_project", SCALE)
    )
    EXPORT_TASKS_PERIODS.append(
        (rec["pressure_class"], f"conversion_pressure_DW_pressure_{period_name}_{label}_project", SCALE)
    )
    

# Current-period wet/dry seasonal composites -- same three
# artifacts per composite as above, labeled current_wet_2022_2025 / current_dry_2022_2025.
CURRENT_SEASONAL_LABELS = {"wet": "current_wet_2022_2025", "dry": "current_dry_2022_2025"}
for season, rec in current_seasonal_records.items():
    label = CURRENT_SEASONAL_LABELS[season]
    EXPORT_TASKS_PERIODS.append(
        (rec["composite"].select(DERIVED_BANDS), f"connectivity_inputs_DW_connectivity_inputs_{label}_project", SCALE)
    )
    
    EXPORT_TASKS_PERIODS.append(
        (rec["habitat_class"], f"connectivity_inputs_DW_class_{label}_project", SCALE)
    )
    EXPORT_TASKS_PERIODS.append(
        (rec["pressure_class"], f"conversion_pressure_DW_pressure_{label}_project", SCALE)
    )
    

print(f"{len(EXPORT_TASKS_PERIODS)} period export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_PERIODS, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} period export tasks.")

### 10d. Transitions & connectivity masks

In [ ]:
EXPORT_TASKS_TRANSITIONS_MASKS = [
    (transitions["baseline_to_current"], "transitions_DW_transition_baseline_2016_2018_to_current_2022_2025_project", SCALE),
    (transitions["pre_to_current"], "transitions_DW_transition_pre_2019_2021_to_current_2022_2025_project", SCALE),
]

CONNECTIVITY_MASK_NAMES = {
    "analysis_mask": "connectivity_inputs_DW_connectivity_analysis_mask_project",
    "natural_habitat_mask": "connectivity_inputs_DW_natural_habitat_mask_current_2022_2025_project",
    "high_quality_source_mask": "connectivity_inputs_DW_high_quality_source_mask_current_2022_2025_project",
}
for mask_name, mask_image in connectivity_masks.items():
    EXPORT_TASKS_TRANSITIONS_MASKS.append((mask_image, CONNECTIVITY_MASK_NAMES[mask_name], SCALE))

print(f"{len(EXPORT_TASKS_TRANSITIONS_MASKS)} transition/mask export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_TRANSITIONS_MASKS, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} transition/mask export tasks.")

## 11. Limitations & next steps

- Dynamic World begins mid-2015, so this is a **2016-present** analysis.
- Classification thresholds (`config.DW_HABITAT_THRESHOLDS`/`config.DW_PRESSURE_THRESHOLDS`) are
  starting values -- calibrate against high-resolution imagery for Enarau, Mbokishi, and the
  corridor before treating any class boundary as final.
- Woodland/shrubland/grassland/bare-ground/cropland confusion is expected in this semi-arid
  savanna landscape, especially at mixed class boundaries or in dry years.
- A real "uncertain" (class 8) fraction is expected in bare-ground-heavy zones -- sand and other
  high-reflectance surfaces lower `top1_prob` by scoring high on multiple DW classes at once;
  this is not necessarily a classification error.
- Cross-check recovery/degradation signals against the Objective 2 productivity/degradation
  (LandTrendr) time series before reporting either as conclusive.
- The connectivity-ready masks and probability stacks exported here are evidence layers only --
  final resistance, source-strength, and corridor products are built in Objective 4.
- This notebook does not run the R `landscapemetrics` step (Objective 3), nor does it read the
  exported CSVs back into pandas -- both consume the Drive exports from this notebook as a
  separate, later step.